# Classification Solutions

Solutions for `exercises.ipynb`. Try the exercises first — peek here only after you've attempted each question.

## Part 1 — Warm-up

**1. Meet the tumours.** 569 samples, 30 numeric scan features, two classes — and the codes are fixed: 0 = malignant, 1 = benign.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer()
X = pd.DataFrame(cancer.data, columns=cancer.feature_names)
y = pd.Series(cancer.target, name="label")

print("Shape:", X.shape)
print("Classes:", {name: int((y == i).sum())
                   for i, name in enumerate(cancer.target_names)})

# sklearn encodes malignant = 0, benign = 1 - knowing which code is which
# class decides what precision/recall mean later.

**2. Fit a classifier.** Despite the name it is a classifier: a linear boundary squashed into probabilities — and it scores well out of the box.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

clf = LogisticRegression(max_iter=10000).fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(f"test accuracy: {clf.score(X_test, y_test):.3f}")
print("predictions:", y_pred[:10])
print("true labels:", y_test[:10].to_numpy())

**3. Confidence, not just verdicts.** `predict_proba` exposes the sigmoid's confidence; column order follows `classes_`, never intuition.

In [ ]:
import numpy as np

proba = clf.predict_proba(X_test)

print("classes:", clf.classes_)
preview = pd.DataFrame({
    "P(benign)": proba[:5, 1].round(3),
    "predicted label": y_pred[:5],
})
print(preview.to_string(index=False))

# Column j of predict_proba is P(class = classes_[j]). Confirm the mapping
# with clf.classes_ BEFORE grabbing a column blindly.

## Part 2 — Practice

**4. Move the threshold.** Lowering the cut catches more true cancers (recall up) at the price of more false alarms — threshold tuning is free policy.

In [ ]:
from sklearn.metrics import recall_score

proba_malignant = clf.predict_proba(X_test)[:, 0]

for t in (0.5, 0.3):
    pred = np.where(proba_malignant >= t, 0, 1)
    flagged = int((pred == 0).sum())
    rec = recall_score(y_test, pred, pos_label=0)
    print(f"threshold {t}: {flagged}/{len(pred)} flagged, "
          f"malignant recall {rec:.3f}")

# A missed cancer (false negative) costs a life; an extra biopsy merely
# costs an afternoon. Clinics slide the threshold down to buy recall.

**5. Map the mistakes.** The four cells give every error a name — and in screening the FN corner is the one that hurts patients.

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

print(f"TP (malignant, caught)        : {tp}")
print(f"FN (malignant, MISSED)        : {fn}")
print(f"FP (benign, false alarm)      : {fp}")
print(f"TN (benign, correct all-clear): {tn}")

# FN - a malignant tumour waved through as benign - is the dangerous
# quadrant; FP merely sends someone for a second look.

**6. Score it by hand, then by sklearn.** Four ratios over four cells — computing them yourself demystifies the report card forever.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

acc = (tp + tn) / cm.sum()
prec = tp / (tp + fp)
rec = tp / (tp + fn)
f1 = 2 * prec * rec / (prec + rec)

print(f"accuracy : {acc:.3f}  ({tp + tn} of {cm.sum()} correct)")
print(f"precision: {prec:.3f}  (when we cry malignant, how often right)")
print(f"recall   : {rec:.3f}  (share of true malignants caught)")
print(f"F1       : {f1:.3f}")

print(classification_report(y_test, y_pred,
                            target_names=["malignant", "benign"]))

**7. Accuracy lies.** Always-legit scores ~95% by agreeing with the imbalance — while recalling exactly nothing from the class that matters.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.datasets import make_classification
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

X_imb, y_imb = make_classification(n_samples=800, n_features=10,
                                   weights=[0.95, 0.05], flip_y=0,
                                   random_state=42)
Xi_tr, Xi_te, yi_tr, yi_te = train_test_split(
    X_imb, y_imb, test_size=0.25, random_state=42, stratify=y_imb)

lazy = DummyClassifier(strategy="most_frequent").fit(Xi_tr, yi_tr)
print(f"do-nothing accuracy: {lazy.score(Xi_te, yi_te):.3f}")
print(classification_report(yi_te, lazy.predict(Xi_te),
                            target_names=["legit", "fraud"], zero_division=0))

# Fraud recall is 0.00 yet the scoreboard glows ~95%. Whenever someone
# brags accuracy, ask: what was the class balance, and what was the rare-
# class recall?

## Part 3 — Challenge

**8. Four classifiers, one referee.** Identical folds make the comparison fair — and unscaled KNN again trails its distance-blind rivals.

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score

zoo = {
    "LogisticRegression": LogisticRegression(max_iter=10000),
    "KNN (k=7)":          KNeighborsClassifier(n_neighbors=7),
    "DecisionTree":       DecisionTreeClassifier(random_state=42),
    "RandomForest":       RandomForestClassifier(n_estimators=100,
                                                 random_state=42),
}
rows = []
for name, model in zoo.items():
    s = cross_val_score(model, X, y, cv=5)
    rows.append({"model": name, "mean_acc": round(s.mean(), 3),
                 "std": round(s.std(), 3)})

board = pd.DataFrame(rows).sort_values("mean_acc", ascending=False)
print(board.to_string(index=False))

# KNN suffers raw units (lesson 02); wrap it with a scaler and watch it
# climb. Same folds for everyone, or the contest is theatre.

**9. Watch a tree overfit live.** Train accuracy climbs toward perfection while test peaks early and sags — deepening trees buy memorisation with generalisation.

In [ ]:
import numpy as np

depths = range(1, 11)
train_scores, test_scores = [], []
for d in depths:
    t = DecisionTreeClassifier(max_depth=d, random_state=42).fit(
        X_train, y_train)
    train_scores.append(t.score(X_train, y_train))
    test_scores.append(t.score(X_test, y_test))

table = pd.DataFrame({"depth": list(depths),
                      "train": np.round(train_scores, 3),
                      "test": np.round(test_scores, 3)})
print(table.to_string(index=False))
best_d = int(np.argmax(test_scores)) + 1
print(f"\nbest test depth: {best_d}")

# Deeper = more branches = more memorised noise: the train curve rises
# monotonically while test crests and declines. The widening gap is
# overfitting made visible.